# 🧠 01 · 图卷积网络 GCN：手写消息传递 + 半监督节点分类

> GCN 是 GNN 的第一课。本节从**拉普拉斯归一化**出发，**手写两层 GCN**：
> 消息传递（聚合邻居）、归一化邻接、ReLU 激活、softmax 分类，
> 并可视化**叠加 K 层=感受野 K 跳邻居**。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | GCN 的聚合公式？ | H' = σ(D̃⁻¹ᐟ² Â D̃⁻¹ᐟ² H W) |
| 2 | 归一化为什么必要？ | 度数差异大时防数值不稳定 |
| 3 | 多层能干什么？ | 层数=聚合跳数（感受野） |
| 4 | 半监督怎么做？ | 部分节点有标签训练 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
import networkx as nx
import sqlite3
import os
rng = np.random.default_rng(42)


## 1️⃣ 构造图与特征

In [2]:
G = nx.karate_club_graph()   # Zachary 空手道俱乐部：34 节点，2 个社区（经典 GNN 入门数据）
nodes = list(G.nodes())
A = nx.to_numpy_array(G)
n, d = len(nodes), 5
# 随机节点特征（真实场景=用户属性/文本向量）
X = rng.normal(0, 1, (n, d))
print('节点数:', n, '边数:', G.number_of_edges(), '特征维度:', d)
# 标签：俱乐部成员划分（1/2），作为分类目标
labels = np.array([1 if G.nodes[i]['club'] == 'Mr. Hi' else 0 for i in nodes])
print('正类(俱乐部1)节点数:', labels.sum(), '负类:', (1 - labels).sum())
assert labels.sum() == 17
print('✅ 数据就绪（真实基准数据集）')

节点数: 34 边数: 78 特征维度: 5
正类(俱乐部1)节点数: 17 负类: 17
✅ 数据就绪（真实基准数据集）


## 2️⃣ 邻接矩阵 + 自环 + 度归一化（核心公式）

In [3]:
# 1) 加自环（让聚合包含自己）
A_hat = A + np.eye(n)
# 2) 对称归一化 D̃⁻¹ᐟ² Â D̃⁻¹ᐟ²
D_hat = np.diag(A_hat.sum(axis=1))
D_inv_sqrt = np.linalg.inv(np.sqrt(D_hat))
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt
print('归一化邻接矩阵形状:', A_norm.shape)
print('每行和（应为 1 附近）:', A_norm.sum(axis=1).round(3)[:5])
print('➡️ 归一化让高低度节点的特征尺度一致，防止高维特征主导')

归一化邻接矩阵形状: (34, 34)
每行和（应为 1 附近）: [1.902 1.378 1.335 1.055 0.775]
➡️ 归一化让高低度节点的特征尺度一致，防止高维特征主导


## 3️⃣ 手写两层 GCN 前向

In [4]:
def gcn_layer(H, A_norm, W):
    return np.tanh(A_norm @ H @ W)

hidden_dim, out_dim = 8, 2
rng2 = np.random.default_rng(7)
W1 = rng2.normal(0, 0.1, (d, hidden_dim))
W2 = rng2.normal(0, 0.1, (hidden_dim, out_dim))

H1 = gcn_layer(X, A_norm, W1)             # 聚合 1 跳
logits = gcn_layer(H1, A_norm, W2)        # 聚合 2 跳
probs = np.exp(logits - logits.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
print('第1层输出:', H1.shape, '| 最终概率:', probs.shape)
# 随机初始化的模型暂时没有意义，但验证形状与数值稳定
assert probs.shape == (n, 2) and np.allclose(probs.sum(axis=1), 1)
print('✅ 两层 GCN 前向正确（softmax 行和为 1）')

第1层输出: (34, 8) | 最终概率: (34, 2)
✅ 两层 GCN 前向正确（softmax 行和为 1）


## 4️⃣ 训练：半监督节点分类

In [5]:
# 半监督：只给 60% 节点标签，训练交叉熵；其余节点做测试
train_mask = np.zeros(n, dtype=bool)
train_mask[rng.choice(n, size=int(n * 0.6), replace=False)] = True

def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def train_gcn(X, A_norm, labels, train_mask, epochs=400, lr=0.05, seed=3):
    r = np.random.default_rng(seed)
    W1 = r.normal(0, 0.1, (d, hidden_dim)); W2 = r.normal(0, 0.1, (hidden_dim, out_dim))
    for ep in range(epochs):
        H1 = np.tanh(A_norm @ X @ W1)
        logits = A_norm @ H1 @ W2
        probs = softmax(logits)
        # 交叉熵（仅训练节点）
        loss = -np.mean(np.log(probs[train_mask][np.arange(train_mask.sum()), labels[train_mask]] + 1e-9))
        # 梯度（简化：softmax 交叉熵梯度 = prob - onehot）
        grad_logits = probs.copy()
        grad_logits[np.arange(n), labels] -= 1
        grad_logits[~train_mask] = 0
        grad_W2 = H1.T @ (A_norm.T @ grad_logits)
        grad_H1 = (A_norm.T @ grad_logits) @ W2.T
        grad_W1 = X.T @ (A_norm.T @ (grad_H1 * (1 - np.tanh(A_norm @ X @ W1) ** 2)))
        W2 -= lr * grad_W2
        W1 -= lr * grad_W1
    return W1, W2

W1t, W2t = train_gcn(X, A_norm, labels, train_mask)
H1t = np.tanh(A_norm @ X @ W1t)
probs_t = softmax(A_norm @ H1t @ W2t)
pred = probs_t.argmax(axis=1)
acc = (pred == labels).mean()
acc_train = (pred[train_mask] == labels[train_mask]).mean()
acc_test = (pred[~train_mask] == labels[~train_mask]).mean()
print(f'总准确率: {acc:.1%} | 训练集: {acc_train:.1%} | 测试集: {acc_test:.1%}')
print('➡️ 只有 60% 节点参与训练，模型通过"邻居聚合"泛化到未标注节点 = 半监督')
assert acc > 0.7
print('✅ 半监督 GCN 分类有效（远高于随机 50%）')

总准确率: 91.2% | 训练集: 95.0% | 测试集: 85.7%
➡️ 只有 60% 节点参与训练，模型通过"邻居聚合"泛化到未标注节点 = 半监督
✅ 半监督 GCN 分类有效（远高于随机 50%）


## 5️⃣ 感受野可视化：K 层 = K 跳邻居

In [6]:
# 从节点 0 出发：1 跳邻居 vs 2 跳邻居
hop1 = set(G.neighbors(0))
hop2 = set()
for nb in hop1:
    hop2 |= set(G.neighbors(nb))
hop2 -= {0}; hop2 -= hop1
print('节点 0 的 1 跳邻居:', sorted(hop1), f'({len(hop1)} 个)')
print('节点 0 的 2 跳邻居:', sorted(hop2), f'({len(hop2)} 个)')
print('''
  消息传递视角：第 k 层节点的表示包含 k 跳内邻居信息；
  所以"层数=感受野"，但层数太深会过平滑（所有节点趋于一致）——GNN 常 2-3 层。
''')
assert 0 not in hop1 and 0 not in hop2
print('✅ 感受野分析完成')

节点 0 的 1 跳邻居: [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 17, 19, 21, 31] (16 个)
节点 0 的 2 跳邻居: [9, 16, 24, 25, 27, 28, 30, 32, 33] (9 个)

  消息传递视角：第 k 层节点的表示包含 k 跳内邻居信息；
  所以"层数=感受野"，但层数太深会过平滑（所有节点趋于一致）——GNN 常 2-3 层。

✅ 感受野分析完成


## 6️⃣ GCN 局限与变体（面试速答）

In [7]:
print('''1) 过平滑：层数深 -> 节点表示趋同 -> 常用 2-3 层；
2) 静态结构：邻接矩阵固定，新增节点/边要重训（GraphSAGE 缓解）；
3) 等方性：所有邻居权重相同（GAT 用注意力加权缓解）；
4) 大规模：整图训练 O(N²) 邻接 -> 邻居采样（GraphSAGE/Cluster-GCN）；
5) 变体：GCN(一阶) / GAT(注意力) / GraphSAGE(采样+聚合) / GIN(图同构)。
''')

1) 过平滑：层数深 -> 节点表示趋同 -> 常用 2-3 层；
2) 静态结构：邻接矩阵固定，新增节点/边要重训（GraphSAGE 缓解）；
3) 等方性：所有邻居权重相同（GAT 用注意力加权缓解）；
4) 大规模：整图训练 O(N²) 邻接 -> 邻居采样（GraphSAGE/Cluster-GCN）；
5) 变体：GCN(一阶) / GAT(注意力) / GraphSAGE(采样+聚合) / GIN(图同构)。



## 📝 自测清单

- [ ] 能写出 GCN 层公式并解释对称归一化
- [ ] 能手写两层 GCN 前向与 softmax
- [ ] 能解释半监督学习的机制（邻居聚合泛化）
- [ ] 能讲清感受野与过平滑问题
- [ ] 能列出 GCN/GAT/GraphSAGE 三个变体各解决什么